# Demucs

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
import torchaudio
from utils.clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [2]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt-origin')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-origin-Demucs')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-Demucs')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load Demucs Model

In [3]:
from demucs.pretrained import get_model
from demucs.apply import apply_model

model_name = 'htdemucs_ft'

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

torch.device(device)

model = get_model(model_name)
model.to(device)
model.eval()

BagOfModels(
  (models): ModuleList(
    (0-3): 4 x HTDemucs(
      (encoder): ModuleList(
        (0): HEncLayer(
          (conv): Conv2d(4, 48, kernel_size=(8, 1), stride=(4, 1), padding=(2, 0))
          (norm1): Identity()
          (rewrite): Conv2d(48, 96, kernel_size=(1, 1), stride=(1, 1))
          (norm2): Identity()
          (dconv): DConv(
            (layers): ModuleList(
              (0): Sequential(
                (0): Conv1d(48, 6, kernel_size=(3,), stride=(1,), padding=(1,))
                (1): GroupNorm(1, 6, eps=1e-05, affine=True)
                (2): GELU(approximate='none')
                (3): Conv1d(6, 96, kernel_size=(1,), stride=(1,))
                (4): GroupNorm(1, 96, eps=1e-05, affine=True)
                (5): GLU(dim=1)
                (6): LayerScale()
              )
              (1): Sequential(
                (0): Conv1d(48, 6, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
                (1): GroupNorm(1, 6, eps=1e-05, affine=Tr

## Denoise Function

In [4]:
def denoise_audio(audio_path, model, device):
    """
    Apply Demucs for source separation and enhancement
    Extract the vocals track from audio, removing background noise, music, etc.
    Note: Demucs classifies all human voices (speech + singing) as vocals

    Args:
        audio_path: Input audio file path
        model: Demucs model instance
        device: Compute device (cuda/mps/cpu)

    Returns:
        vocals_audio: Extracted vocals audio numpy array (mono)
        sr: Sample rate
    """
    # Load audio
    audio, sr = sf.read(str(audio_path))

    # Demucs requires 2-channel (stereo) input
    # Handle audio with different channel counts
    if len(audio.shape) == 1:
        # Mono: duplicate to stereo [time] -> [time, 2]
        audio = np.stack([audio, audio], axis=1)
    elif len(audio.shape) == 2:
        if audio.shape[1] == 1:
            # [time, 1] -> [time, 2]
            audio = np.concatenate([audio, audio], axis=1)
        elif audio.shape[1] > 2:
            # More than 2 channels, take only the first 2
            audio = audio[:, :2]
        # If already 2 channels, leave unchanged

    # Resample to the model's required sample rate
    target_sr = model.samplerate
    if sr != target_sr:
        # Resample each channel separately
        num_samples = int(audio.shape[0] * target_sr / sr)
        audio_resampled = np.zeros((num_samples, 2), dtype=audio.dtype)
        for ch in range(2):
            audio_resampled[:, ch] = signal.resample(audio[:, ch], num_samples)
        audio = audio_resampled
        sr = target_sr

    # Convert to torch tensor
    # Demucs expects input shape [batch, channels, time]
    # audio current shape: [time, 2]
    audio_tensor = torch.from_numpy(audio.T).float()  # [2, time]
    audio_tensor = audio_tensor.unsqueeze(0)  # [1, 2, time]
    audio_tensor = audio_tensor.to(device)

    # Apply Demucs source separation
    with torch.no_grad():
        # apply_model returns the separated audio sources
        # Output shape: [batch, sources, channels, time]
        # sources order is typically: ['drums', 'bass', 'other', 'vocals']
        sources = apply_model(
            model,
            audio_tensor,
            device=device,
            split=True,  # Process in chunks to save memory
            overlap=0.25  # 25% overlap to avoid boundary artifacts
        )

    # Extract the vocals source (contains all human voice: speech + singing)
    # Find the index of vocals in the sources list
    try:
        vocals_idx = model.sources.index('vocals')
    except (AttributeError, ValueError):
        # If not found, assume it is the last source
        vocals_idx = -1

    # Convert back to numpy and downmix to mono
    # sources shape: [batch, sources, channels, time]
    # Take the average of the two channels as mono output
    vocals_stereo = sources[0, vocals_idx, :, :].cpu().numpy()  # [2, time]
    vocals_audio = np.mean(vocals_stereo, axis=0)  # [time] mono

    return vocals_audio, sr

In [5]:
def batch_denoise(files, output_subdir, model, device, group_name):
    """
    Batch source separation (clears memory before and after each file)
    Extract the vocals track and remove background noise

    Args:
        files: List of audio files to process
        output_subdir: Output subdirectory
        model: Demucs model instance
        device: Compute device
        group_name: Group name (for progress display)
    """
    # Create output directory
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        # Unify output as .wav format
        output_file = output_subdir / (audio_file.stem + '.wav')

        # Skip already processed files
        if output_file.exists():
            skip_count += 1
            continue

        try:
            # ⚡ Clear memory before processing
            clear_memory()

            # Source separation, extract vocals
            vocals_audio, sr = denoise_audio(audio_file, model, device)

            # Save (16-bit integer format)
            sf.write(str(output_file), vocals_audio, sr, subtype='PCM_16')
            success_count += 1

            # ⚡ Clear memory immediately after processing
            del vocals_audio  # Free large array
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\n✗ Failed: {audio_file.name}: {e}")
            # ⚡ Clear memory after failure as well
            clear_memory()

    # Print statistics
    print(f"\n{group_name} processing complete:")
    print(f"  Success: {success_count}")
    print(f"  Skipped: {skip_count}")
    print(f"  Failed: {fail_count}")

## Pitt Denoise

In [6]:
batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    model,
    device,
    group_name='Dementia'
)

# Clear memory between group s
clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    model,
    device,
    group_name='Control'
)

Processing Dementia: 100%|██████████| 309/309 [19:13<00:00,  3.73s/it]



Dementia 处理完成:
  成功: 309
  跳过: 0
  失败: 0


Processing Control: 100%|██████████| 243/243 [23:26<00:00,  5.79s/it]


Control 处理完成:
  成功: 243
  跳过: 0
  失败: 0


## Lu Denoise

In [ ]:
batch_denoise(
    dementia_files_Lu,
    output_dir_Lu / 'Dementia',
    model,
    device,
    group_name='Dementia'
)

# Clear memory between group s
clear_memory()

batch_denoise(
    control_files_Lu,
    output_dir_Lu / 'Control',
    model,
    device,
    group_name='Control'
)